[![../img/cloudevel.png](../img/cloudevel.png)](https://cloudevel.com)

# Introducción.

**sql101 · SQL para Ingeniería y Analítica de Datos** · Cloudevel® · Instructor: José Luis Chiquete · Módulo 1: *Fundamentos*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se presentan el papel de *SQL* en la ingeniería y la analítica de datos, el estándar *ANSI SQL* y su importancia, el motor del taller (*DuckDB*) y sus alternativas, el entorno de trabajo y los datos.

## 🧭 Cómo usar este *notebook*.

* Ejecuta primero la celda de **configuración** y después las demás **en orden** (```Shift + Enter```).
* Las celdas que comienzan con ```%%sql``` son consultas: todo lo que sigue a ```%%sql``` es *SQL* y puede copiarse a otro motor.
* **Ejercicio:** lo que todos deben dominar. **Reto (opcional):** para quien quiere profundizar; se puede resolver como práctica.
* En las celdas ```-- TU CÓDIGO AQUÍ```, reemplaza la línea ```SELECT NULL AS tu_respuesta;``` por tu consulta. Las de *✍️ Tu respuesta* son para escribir.

## Configuración.

Abre la base de datos del taller en modo de sólo lectura y habilita las celdas ```%%sql```.

In [ ]:
import duckdb

%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.displaycon = False

conexion = duckdb.connect('../data/thelook.duckdb', read_only=True)
%sql conexion

## *SQL* en ingeniería y analítica de datos.

*SQL* (*Structured Query Language*) es un lenguaje declarativo para consultar y transformar datos estructurados. En lugar de indicar *cómo* obtener un resultado, se describe *qué* resultado se desea y el motor de la base de datos decide la forma más eficiente de calcularlo.

En el ciclo de vida de los datos, *SQL* aparece en prácticamente todas las etapas:

* **Analítica de datos:** responder preguntas de negocio, construir indicadores (*KPIs*) y alimentar tableros.
* **Ingeniería de datos:** transformar datos crudos en tablas limpias y confiables, deduplicar, integrar fuentes y cargar datos de forma incremental.

## *ANSI SQL*.

*SQL* nació en *IBM* a principios de los años setenta y fue estandarizado por *ANSI* en 1986 y por *ISO* en 1987. Desde entonces, el estándar (*ISO/IEC 9075*) se ha revisado varias veces, incorporando capacidades que hoy se usan a diario:

| Versión | Aportaciones relevantes para este taller |
|---|---|
| *SQL-92* | ```JOIN``` explícitos (```INNER```, ```LEFT```, ```FULL```), ```CASE```, ```CAST```, ```COALESCE```, ```NULLIF```, tipos ```DATE``` y ```TIMESTAMP``` |
| *SQL:1999* | Expresiones de tabla comunes (```WITH```), consultas recursivas |
| *SQL:2003* | Funciones de ventana (```OVER```), ```MERGE```, ```CREATE TABLE ... AS``` |
| *SQL:2008* | ```FETCH FIRST n ROWS ONLY```, ```TRUNCATE``` |
| *SQL:2011* | ```LAG```, ```LEAD``` y datos temporales |
| *SQL:2016* en adelante | Funciones *JSON*, reconocimiento de patrones en filas |

Ningún motor implementa el estándar completo, y todos agregan extensiones propias. A cada variante se le llama **dialecto**: *GoogleSQL* en *BigQuery*, *T-SQL* en *SQL Server*, el *SQL* de *PostgreSQL* o el de *DuckDB*, por ejemplo.

### ¿Por qué apegarse al estándar?

* **Transferibilidad:** en ingeniería y analítica de datos es común trabajar con varios motores a la vez (una base transaccional, un *data warehouse*, un motor local para prototipos). El *SQL* estándar funciona en todos con cambios mínimos.
* **Vigencia:** los dialectos cambian y los proveedores se reemplazan; el estándar perdura.
* **Legibilidad:** cualquier persona que conozca *SQL* puede leer y mantener una consulta estándar.
* **Migración:** mover procesos entre plataformas (por ejemplo, de un servidor local a la nube) cuesta menos cuando no dependen de extensiones.

**Este taller se apega a *ANSI SQL*.** Cuando un motor obliga a usar otra sintaxis, o cuando vale la pena conocer una extensión, se indica con una nota:

> **Nota de dialecto:** descripción de la diferencia con el estándar.

## El motor del taller: *DuckDB*.

[*DuckDB*](https://duckdb.org) es un motor de bases de datos analítico (*OLAP*) que se ejecuta dentro del propio proceso, sin servidor, y guarda una base de datos completa en un solo archivo.

Se eligió por las siguientes razones:

* **Apego al estándar:** implementa la mayor parte de *ANSI SQL* que se estudia en el taller (tipos de fecha reales, ```EXTRACT```, funciones de ventana, ```MERGE```, esquemas) y rechaza construcciones inválidas en lugar de aceptarlas en silencio.
* **Orientado a analítica:** almacena los datos por columnas, igual que *BigQuery*, *Snowflake* o *Redshift*, por lo que lo aprendido sobre rendimiento se traslada a un *data warehouse* real.
* **Sin instalación ni costo:** se instala con ```pip```, funciona dentro de *GitHub Codespaces* y no requiere cuentas, tarjetas de crédito ni conexión a la nube.
* **Herramienta real de ingeniería de datos:** se usa en producción para procesar archivos *CSV* y *Parquet*, en *pipelines* con *dbt* y como motor local de prototipos.

### Alternativas consideradas.

| Motor | Tipo | Ventajas | Por qué no es el motor principal |
|---|---|---|---|
| ***SQLite*** | Embebido, transaccional (*OLTP*) | Viene incluido en *Python*; el motor más usado del mundo | No tiene tipos de fecha reales (no admite ```EXTRACT``` ni literales ```DATE```), su tipado es dinámico, acepta ```GROUP BY``` inválidos y no tiene ```MERGE``` |
| ***PostgreSQL*** | Servidor, transaccional | Muy apegado al estándar; referencia de la industria | Requiere un servidor; está optimizado para transacciones, no para analítica |
| ***BigQuery*** | *Data warehouse* en la nube | El *data warehouse* de *Google Cloud*; escala a *petabytes* | Requiere cuenta de *Google Cloud*; cobra por bytes procesados; el *sandbox* no permite *DML* |
| ***Snowflake*** / ***Redshift*** | *Data warehouse* en la nube | Estándar en muchas empresas | Requieren cuenta de pago |
| ***MySQL*** | Servidor, transaccional | Muy popular en aplicaciones *web* | Menos apegado al estándar; orientado a transacciones |

## Entorno de trabajo.

El taller se ejecuta en un **contenedor de desarrollo** (*devcontainer*), que puede abrirse en *GitHub Codespaces* o localmente con *VS Code* y *Docker*. Al crearse, el contenedor:

1. Instala *DuckDB* y [*JupySQL*](https://jupysql.ploomber.io), una extensión de *Jupyter* que permite escribir *SQL* directamente en las celdas.
2. Genera los datos del taller con ```scripts/generar_datos.py```.
3. Los carga en la base ```data/thelook.duckdb``` con ```scripts/cargar_datos.py```.

Para restablecer los datos en cualquier momento, ejecuta en la terminal:

```
python scripts/cargar_datos.py
```

### Celdas de *SQL*.

Cada notebook comienza con una celda de configuración en *Python*, que se ejecuta una sola vez:

* ```%load_ext sql``` habilita *JupySQL*.
* ```SqlMagic.displaylimit = 0``` indica que se muestre el resultado completo de cada consulta, sin recortes de la herramienta.
* ```duckdb.connect(..., read_only=True)``` abre la base en modo de sólo lectura, lo que permite tener varios notebooks abiertos al mismo tiempo.

A partir de ahí, cualquier celda que comience con ```%%sql``` se ejecuta como una consulta y muestra su resultado como tabla. **Todo lo que sigue a ```%%sql``` es *SQL* estándar** y puede copiarse a cualquier otro motor.
%%sql``` se ejecuta como una consulta y muestra su resultado como tabla.

## El conjunto de datos: *TheLook eCommerce*.

Los datos simulan una tienda de ropa en línea y replican el esquema del *dataset* público [```thelook_ecommerce```](https://console.cloud.google.com/marketplace/product/bigquery-public-data/thelook-ecommerce) de *BigQuery*, de modo que las consultas del taller pueden ejecutarse también allí. Los datos son sintéticos: las marcas y las personas son ficticias.

| Tabla | Contenido | Clave | Se relaciona con |
|---|---|---|---|
| ```users``` | Clientes | ```id``` | |
| ```orders``` | Órdenes de compra | ```order_id``` | ```users``` mediante ```user_id``` |
| ```order_items``` | Artículos de cada orden | ```id``` | ```orders``` (```order_id```), ```products``` (```product_id```), ```users``` (```user_id```) |
| ```products``` | Catálogo de productos | ```id``` | ```distribution_centers``` mediante ```distribution_center_id``` |
| ```distribution_centers``` | Centros de distribución | ```id``` | |

## Primeras consultas.

* La siguiente declaración no consulta ninguna tabla; sólo evalúa expresiones. Observa que **las cadenas de texto se escriben con comillas simples**.

In [ ]:
%%sql
SELECT 'Hola, SQL' AS saludo, 2 + 3 AS suma;

* El estándar define el esquema ```INFORMATION_SCHEMA```, con vistas que describen los objetos de la base de datos. Las siguientes consultas listan las tablas disponibles y las columnas de ```products```.

In [ ]:
%%sql
SELECT table_name
FROM information_schema.tables
ORDER BY table_name;

In [ ]:
%%sql
SELECT column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_name = 'products'
ORDER BY ordinal_position;

* El identificador, nombre y país de los primeros 10 clientes. La tabla tiene miles de filas; la cláusula estándar ```FETCH FIRST``` indica cuántas regresar (se estudia a detalle en la siguiente lección).

In [ ]:
%%sql
SELECT id, first_name, country
FROM users
ORDER BY id
FETCH FIRST 10 ROWS ONLY;

## ✏️ Ejercicios.

Los **ejercicios** cubren lo que todos deben dominar; los **retos** son opcionales. Las soluciones están en ```soluciones/```.

### Ejercicio 1 · Columnas de una tabla.

Consulta ```information_schema.columns``` para obtener las columnas de ```orders``` y su tipo de dato. ¿Qué columnas son de tipo ```TIMESTAMP```?

**💡 Solución.** Columnas de ```orders``` y su tipo. Son ```TIMESTAMP```: ```created_at```, ```returned_at```, ```shipped_at``` y ```delivered_at```.

In [ ]:
%%sql
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_catalog = 'thelook' AND table_name = 'orders'
ORDER BY ordinal_position;

### Ejercicio 2 · Contar las tablas.

Obtén el número de tablas que hay en la base de datos.

**💡 Solución.** Número de tablas.

In [ ]:
%%sql
SELECT COUNT(*) AS tablas
FROM information_schema.tables
WHERE table_catalog = 'thelook' AND table_type = 'BASE TABLE';

### Ejercicio 3 · Primera consulta propia.

Escribe una consulta que regrese el ```id```, ```name``` y ```brand``` de los primeros 20 productos, ordenados por ```id```.

In [ ]:
%%sql
SELECT id, name, brand
FROM products
ORDER BY id
FETCH FIRST 20 ROWS ONLY;

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>